<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Cuadernos/06_BoW_NGramas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Bag-of-Words y N-gramas</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Introducción</span>](#introduccion)
- [<span style="color: #005C99;">Bag-of-Words y sus límites</span>](#bag-of-words-y-sus-limites)
- [<span style="color: #005C99;">N-gramas como características</span>](#n-gramas-como-caracteristicas)
- [<span style="color: #005C99;">Modelos de lenguaje</span>](#modelos-de-lenguaje)
- [<span style="color: #005C99;">Estimación por máxima verosimilitud</span>](#estimacion-por-maxima-verosimilitud)
- [<span style="color: #005C99;">El problema de los ceros y las palabras desconocidas</span>](#el-problema-de-los-ceros-y-las-palabras-desconocidas)
- [<span style="color: #005C99;">Suavizado</span>](#suavizado)
- [<span style="color: #005C99;">Perplejidad</span>](#perplejidad)
- [<span style="color: #005C99;">Generación de texto</span>](#generacion-de-texto)
- [<span style="color: #005C99;">Referencias</span>](#referencias)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import cess_esp
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

_ = nltk.download("cess_esp", quiet=True)

<a id="introduccion"></a>
<h2 style="color: #007ACC;">Introducción</h2>

En el módulo 1 representamos los textos como **bolsas de palabras**: vectores de conteos donde el orden no importa. Funcionó sorprendentemente bien para clasificar sentimiento y temas, pero tiene un límite evidente: el lenguaje **es una secuencia**.

En este módulo pasamos a modelar secuencias con **probabilidades**. Empezamos por la pregunta más básica:

> ¿Qué tan probable es una secuencia de palabras? ¿Y cuál es la siguiente palabra más probable?

Un **modelo de lenguaje** responde exactamente eso. Es la base del autocompletado del teclado, de la corrección ortográfica, del reconocimiento de voz y de la traducción automática. Los grandes modelos de lenguaje (GPT, Claude...) son, en el fondo, modelos que predicen la siguiente palabra; los **N-gramas** son su versión clásica, más simple y totalmente interpretable.

<a id="bag-of-words-y-sus-limites"></a>
<h2 style="color: #007ACC;">Bag-of-Words y sus límites</h2>

Recordemos el modelo de bolsa de palabras con `CountVectorizer`. Dos oraciones con significados opuestos pueden tener **exactamente el mismo vector**:

In [3]:
oraciones = ["el perro muerde al hombre", "el hombre muerde al perro"]
bow = CountVectorizer().fit(oraciones)
X = bow.transform(oraciones).toarray()

print(pd.DataFrame(X, columns=bow.get_feature_names_out(), index=oraciones))
print("\n¿Vectores idénticos?", (X[0] == X[1]).all())

                           al  el  hombre  muerde  perro
el perro muerde al hombre   1   1       1       1      1
el hombre muerde al perro   1   1       1       1      1

¿Vectores idénticos? True


Otras variantes de la bolsa de palabras:

- **Binaria** (`binary=True`): solo registra presencia o ausencia. Suele funcionar mejor con documentos cortos o con Naive Bayes de Bernoulli.
- **Conteos** (por defecto).
- **TF-IDF** (cuaderno 05).

Todas comparten el mismo supuesto: el orden de las palabras se pierde.

<a id="n-gramas-como-caracteristicas"></a>
<h2 style="color: #007ACC;">N-gramas como características</h2>

Un **N-grama** es una secuencia de $N$ tokens consecutivos:

| N | Nombre | Ejemplos en *"el perro muerde al hombre"* |
|---|--------|----------------------------------------------|
| 1 | unigrama | *el*, *perro*, *muerde*... |
| 2 | bigrama | *el perro*, *perro muerde*, *muerde al*... |
| 3 | trigrama | *el perro muerde*, *perro muerde al*... |

Usar N-gramas como características recupera parte del orden local: *"perro muerde"* y *"hombre muerde"* ya son dimensiones distintas.

In [4]:
bow12 = CountVectorizer(ngram_range=(1, 2)).fit(oraciones)
X12 = bow12.transform(oraciones).toarray()
print(pd.DataFrame(X12, columns=bow12.get_feature_names_out(), index=oraciones).T)
print("\n¿Vectores idénticos?", (X12[0] == X12[1]).all())

               el perro muerde al hombre  el hombre muerde al perro
al                                     1                          1
al hombre                              1                          0
al perro                               0                          1
el                                     1                          1
el hombre                              0                          1
el perro                               1                          0
hombre                                 1                          1
hombre muerde                          0                          1
muerde                                 1                          1
muerde al                              1                          1
perro                                  1                          1
perro muerde                           1                          0

¿Vectores idénticos? False


El precio es el **tamaño del vocabulario**, que crece muy rápido con $N$. Usemos el corpus **CESS-ESP** de NLTK: unas 6 000 oraciones de noticias en español (≈190 000 palabras).

In [5]:
# El corpus marca los sujetos omitidos con "*0*" (los eliminamos) y escribe algunos signos
# de puntuación como códigos: -Fpa- y -Fpt- son paréntesis y -fe- son comillas
SIGNOS = {"-fpa-": "(", "-fpt-": ")", "-fe-": '"'}
oraciones_cess = [[SIGNOS.get(w.lower(), w.lower()) for w in s if not w.startswith("*")] for s in cess_esp.sents()]
print(f"Oraciones: {len(oraciones_cess)}")
print(" ".join(oraciones_cess[1]))

Oraciones: 6030
una portavoz de edf explicó a efe que el proyecto para la construcción de altamira_2 , al norte de tampico , prevé la utilización de gas natural como combustible principal en una central de ciclo combinado que debe empezar a funcionar en mayo_del_2002 .


Fíjate en tokens como `altamira_2` o `mayo_del_2002`: CESS-ESP une con guiones bajos las **expresiones de varias palabras** (nombres propios, fechas, locuciones como `al_fin`) y las trata como un solo token.

In [6]:
def ngramas(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


filas = []
for n in range(1, 6):
    conteo = Counter(g for s in oraciones_cess for g in ngramas(s, n))
    filas.append({"N": n, "N-gramas distintos": len(conteo),
                  "aparecen una sola vez": f"{sum(c == 1 for c in conteo.values()) / len(conteo):.0%}"})
pd.DataFrame(filas).set_index("N")

,N-gramas distintos,aparecen una sola vez
N,,
1,24494,56%
2,99957,80%
3,153690,92%
4,165214,98%
5,162723,99%


Con $N = 3$ la gran mayoría de los trigramas aparece **una sola vez**. Esto es la **dispersión de datos** (*data sparsity*): mientras más largo el N-grama, más informativo, pero también más difícil de estimar con datos finitos. Este compromiso guiará todo el cuaderno.

<a id="modelos-de-lenguaje"></a>
<h2 style="color: #007ACC;">Modelos de lenguaje</h2>

Un **modelo de lenguaje** asigna una probabilidad a cada secuencia de palabras $w_1, w_2, \dots, w_n$. Por la **regla de la cadena** de la probabilidad:

$$P(w_1, \dots, w_n) = P(w_1)\, P(w_2 \mid w_1)\, P(w_3 \mid w_1, w_2) \cdots = \prod_{i=1}^{n} P(w_i \mid w_1, \dots, w_{i-1})$$

El problema es que la historia $w_1, \dots, w_{i-1}$ es casi siempre única: no podemos estimar esas probabilidades contando. La solución es el **supuesto de Markov**: la siguiente palabra solo depende de las $N-1$ anteriores.

$$P(w_i \mid w_1, \dots, w_{i-1}) \approx P(w_i \mid w_{i-N+1}, \dots, w_{i-1})$$

Para un modelo de **bigramas** ($N = 2$):

$$P(w_1, \dots, w_n) \approx \prod_{i=1}^{n} P(w_i \mid w_{i-1})$$

Añadimos marcadores especiales de inicio `<s>` y fin `</s>` de oración. Así el modelo aprende qué palabras suelen empezar oraciones y cuándo terminarlas, y las probabilidades de todas las oraciones posibles suman 1.

<a id="estimacion-por-maxima-verosimilitud"></a>
<h2 style="color: #007ACC;">Estimación por máxima verosimilitud</h2>

La estimación más natural es la de **máxima verosimilitud** (MLE): contar y normalizar.

$$P_{\text{MLE}}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}$$

Primero separamos las oraciones en entrenamiento (90 %) y prueba (10 %). Evaluar un modelo de lenguaje con el mismo texto con el que se entrenó es tan engañoso como en cualquier otro modelo.

In [7]:
rng = random.Random(SEED)
indices = list(range(len(oraciones_cess)))
rng.shuffle(indices)
corte = int(0.9 * len(indices))
train = [oraciones_cess[i] for i in indices[:corte]]
test = [oraciones_cess[i] for i in indices[corte:]]
print(f"Entrenamiento: {len(train)} oraciones | Prueba: {len(test)} oraciones")


def con_marcadores(oracion, n):
    return ["<s>"] * (n - 1) + oracion + ["</s>"]


conteo_bi = Counter(g for s in train for g in ngramas(con_marcadores(s, 2), 2))
conteo_uni = Counter(w for s in train for w in con_marcadores(s, 2))


def p_mle(palabra, previa):
    return conteo_bi[(previa, palabra)] / conteo_uni[previa]


for previa in ["el", "presidente", "millones"]:
    siguientes = Counter({w: c for (p, w), c in conteo_bi.items() if p == previa})
    print(f"P(· | {previa!r}):", ", ".join(f"{w} ({c / conteo_uni[previa]:.3f})" for w, c in siguientes.most_common(5)))

Entrenamiento: 5427 oraciones | Prueba: 603 oraciones
P(· | 'el'): presidente (0.021), que (0.021), gobierno (0.014), primer (0.010), mundo (0.009)
P(· | 'presidente'): de (0.338), del (0.182), , (0.066), . (0.025), estadounidense (0.025)
P(· | 'millones'): de (0.866), en (0.038), , (0.032), . (0.016), - (0.005)


Estas distribuciones ya capturan regularidades del español periodístico: después de *millones* viene *de* (*millones de pesetas*), y después de *presidente* vienen *del* o *de*. Podemos calcular la probabilidad de una oración entera. Para evitar el *underflow* sumamos logaritmos:

In [8]:
def log_prob_oracion(oracion, p):
    tokens = con_marcadores(oracion, 2)
    total = 0.0
    for previa, palabra in ngramas(tokens, 2):
        prob = p(palabra, previa)
        if prob == 0:
            return float("-inf")
        total += math.log(prob)
    return total


for frase in ["el presidente del gobierno .", "el presidente del parlamento galáctico ."]:
    print(f"{frase:<42} log P = {log_prob_oracion(frase.split(), p_mle):.2f}")

el presidente del gobierno .               log P = -14.83
el presidente del parlamento galáctico .   log P = -inf


<a id="el-problema-de-los-ceros-y-las-palabras-desconocidas"></a>
<h2 style="color: #007ACC;">El problema de los ceros y las palabras desconocidas</h2>

La segunda oración recibe probabilidad **cero** (log P = −∞), solo porque *parlamento galáctico* nunca apareció en el entrenamiento. Este es el gran problema del MLE:

1. **N-gramas no vistos:** cualquier bigrama ausente del entrenamiento tiene probabilidad 0, y anula la oración completa.
2. **Palabras desconocidas** (fuera del vocabulario, OOV): ni siquiera tienen un conteo de unigrama.

¿Qué tan frecuente es esto en el conjunto de prueba?

In [9]:
vocab_train = set(conteo_uni)
bigramas_test = [g for s in test for g in ngramas(con_marcadores(s, 2), 2)]
palabras_test = [w for s in test for w in s]

print(f"Palabras de prueba fuera del vocabulario: {np.mean([w not in vocab_train for w in palabras_test]):.1%}")
print(f"Bigramas de prueba nunca vistos:          {np.mean([g not in conteo_bi for g in bigramas_test]):.1%}")
print(f"Oraciones de prueba con probabilidad 0:   {np.mean([log_prob_oracion(s, p_mle) == float('-inf') for s in test]):.1%}")

Palabras de prueba fuera del vocabulario: 8.1%
Bigramas de prueba nunca vistos:          42.3%
Oraciones de prueba con probabilidad 0:   97.8%


¡Casi todas las oraciones de prueba son "imposibles" para el modelo MLE! Lo resolvemos en dos pasos:

**1. Palabras desconocidas: el token `<UNK>`.** Fijamos un vocabulario (por ejemplo, las palabras que aparecen al menos 2 veces en entrenamiento) y reemplazamos todas las demás por `<UNK>`, tanto en entrenamiento como en prueba. Así el modelo aprende la probabilidad de "una palabra rara" en cada contexto.

In [10]:
frec = Counter(w for s in train for w in s)
VOCAB = {w for w, c in frec.items() if c >= 2} | {"</s>", "<UNK>"}


def unkificar(oracion):
    return [w if w in VOCAB else "<UNK>" for w in oracion]


train_u = [unkificar(s) for s in train]
test_u = [unkificar(s) for s in test]
print(f"Tamaño del vocabulario: {len(VOCAB)}")
print(" ".join(test_u[0]))

Tamaño del vocabulario: 9927
la primera medida del gobierno ha sido <UNK> a los directores de las <UNK> a <UNK> " de forma rápida " de los accidentes <UNK> y a no <UNK> en las investigaciones de la policía local .


**2. N-gramas no vistos: suavizado.** Es el tema de la siguiente sección.

<a id="suavizado"></a>
<h2 style="color: #007ACC;">Suavizado</h2>

El **suavizado** le quita un poco de masa de probabilidad a los eventos vistos para repartirla entre los no vistos.

**Suavizado aditivo (Laplace / add-k):** igual que en Naive Bayes (cuaderno 04), sumamos $k$ a todos los conteos:

$$P_{\text{add-}k}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i) + k}{C(w_{i-1}) + k|V|}$$

Con $k = 1$ (Laplace) se reparte demasiada masa a los bigramas no vistos, porque hay muchísimos: con $|V| \approx 9\,000$ hay unos 80 millones de bigramas posibles. Valores pequeños de $k$ funcionan mucho mejor.

**Interpolación lineal:** combina modelos de distintos órdenes. Si el bigrama es poco confiable, el unigrama aporta información:

$$P_{\text{interp}}(w_i \mid w_{i-2}, w_{i-1}) = \lambda_3 P(w_i \mid w_{i-2}, w_{i-1}) + \lambda_2 P(w_i \mid w_{i-1}) + \lambda_1 P(w_i), \qquad \sum_j \lambda_j = 1$$

Los $\lambda$ se eligen sobre datos de validación, no de prueba.

Existen técnicas más sofisticadas, como el **suavizado de Kneser-Ney**, que durante años fue el estándar en reconocimiento de voz y traducción. NLTK lo implementa en `nltk.lm`.

Implementemos una clase general:

In [11]:
class ModeloNGramas:
    """Modelo de lenguaje de N-gramas con suavizado add-k."""

    def __init__(self, n, k=1.0):
        self.n, self.k = n, k

    def entrenar(self, oraciones):
        self.conteos = Counter()   # conteos de N-gramas
        self.contextos = Counter()  # conteos de los contextos (N-1 gramas)
        for s in oraciones:
            for g in ngramas(con_marcadores(s, self.n), self.n):
                self.conteos[g] += 1
                self.contextos[g[:-1]] += 1
        self.V = len(VOCAB)
        return self

    def prob(self, palabra, contexto):
        contexto = tuple(contexto)
        return (self.conteos[contexto + (palabra,)] + self.k) / (self.contextos[contexto] + self.k * self.V)

    def log_prob_oracion(self, oracion):
        tokens = con_marcadores(oracion, self.n)
        return sum(math.log(self.prob(g[-1], g[:-1])) for g in ngramas(tokens, self.n))


m = ModeloNGramas(2, k=1.0).entrenar(train_u)
for frase in ["el presidente del gobierno .", "gobierno del . presidente el", "el presidente del parlamento galáctico ."]:
    print(f"{frase:<42} log P = {m.log_prob_oracion(unkificar(frase.split())):.2f}")

el presidente del gobierno .               log P = -27.34
gobierno del . presidente el               log P = -55.35
el presidente del parlamento galáctico .   log P = -32.89


Ahora la oración con *galáctico* (convertido en `<UNK>`) tiene una probabilidad finita. Además, la oración con las mismas palabras desordenadas es mucho menos probable que la original: a diferencia de la bolsa de palabras, el modelo de bigramas sí distingue el orden. Para comparar modelos necesitamos una métrica.

<a id="perplejidad"></a>
<h2 style="color: #007ACC;">Perplejidad</h2>

La **perplejidad** (PP) es la métrica estándar de los modelos de lenguaje. Para un conjunto de prueba con $N$ tokens:

$$\text{PP}(W) = P(w_1, \dots, w_N)^{-1/N} = \exp\left( -\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid \text{historia}) \right)$$

- Es el inverso de la probabilidad del texto, normalizado por su longitud. **Menor es mejor**.
- Se interpreta como un **factor de ramificación promedio**: una perplejidad de 100 significa que el modelo está tan indeciso como si eligiera uniformemente entre 100 palabras en cada paso.
- Un modelo que no aprende nada (distribución uniforme sobre el vocabulario) tiene perplejidad $|V|$.
- Solo es comparable entre modelos con **el mismo vocabulario** y el mismo tratamiento de `<UNK>`.

In [12]:
def perplejidad(modelo, oraciones):
    log_p = sum(modelo.log_prob_oracion(s) for s in oraciones)
    n_tokens = sum(len(s) + 1 for s in oraciones)  # +1 por </s>
    return math.exp(-log_p / n_tokens)


resultados = []
for n in [1, 2, 3]:
    for k in [1.0, 0.1, 0.01, 0.001]:
        modelo = ModeloNGramas(n, k).entrenar(train_u)
        resultados.append({"N": n, "k": k, "perplejidad": perplejidad(modelo, test_u)})

tabla = pd.DataFrame(resultados).pivot(index="k", columns="N", values="perplejidad").round(1)
tabla.columns = [f"{n}-grama" for n in tabla.columns]
print(f"Referencia: modelo uniforme = {len(VOCAB)}")
tabla

Referencia: modelo uniforme = 9927


,1-grama,2-grama,3-grama
k,,,
0.001,306.2,257.1,1124.9
0.010,306.3,256.5,1222.0
0.100,306.6,369.7,1964.5
1.000,310.7,823.8,3920.2


Observaciones:

- Todos los modelos son muchísimo mejores que el uniforme.
- El suavizado de Laplace ($k = 1$) es muy malo para bigramas y trigramas: regala demasiada probabilidad a combinaciones imposibles. Con $k$ pequeño los bigramas superan claramente a los unigramas.
- Los trigramas con add-k **no** mejoran a los bigramas: hay tan pocos datos (≈170 000 palabras) que casi todos los contextos de dos palabras son nuevos. Este es el compromiso de la dispersión de datos.

> ⚠️ Aquí elegimos $k$ mirando el conjunto de prueba para ilustrar la tendencia. En un experimento real, $k$ (y los $\lambda$ de la interpolación) se eligen sobre un conjunto de **validación** separado.

La interpolación permite que el trigrama aporte cuando tiene información sin que lo perjudiquen los contextos nuevos:

In [13]:
class ModeloInterpolado:
    """Interpolación lineal de unigramas, bigramas y trigramas (estimados por MLE)."""

    def __init__(self, lambdas):
        self.lambdas = lambdas  # (λ1, λ2, λ3)

    def entrenar(self, oraciones):
        self.modelos = [ModeloNGramas(n, k=0.0).entrenar(oraciones) for n in (1, 2, 3)]
        self.modelos[0].k = 1e-3  # el unigrama necesita algo de suavizado para cubrir todo el vocabulario
        return self

    def prob(self, palabra, contexto):
        p = 0.0
        for lam, mod in zip(self.lambdas, self.modelos):
            ctx = tuple(contexto[len(contexto) - (mod.n - 1):]) if mod.n > 1 else ()
            if mod.n == 1 or mod.contextos[ctx] > 0:  # sin contexto visto, ese orden no aporta
                p += lam * mod.prob(palabra, ctx)
        return p

    def log_prob_oracion(self, oracion):
        tokens = con_marcadores(oracion, 3)
        return sum(math.log(self.prob(g[-1], g[:-1])) for g in ngramas(tokens, 3))


for lambdas in [(0.3, 0.7, 0.0), (0.2, 0.5, 0.3), (0.1, 0.5, 0.4), (0.1, 0.3, 0.6)]:
    modelo = ModeloInterpolado(lambdas).entrenar(train_u)
    print(f"λ = {lambdas}: perplejidad = {perplejidad(modelo, test_u):.1f}")

λ = (0.3, 0.7, 0.0): perplejidad = 138.8


λ = (0.2, 0.5, 0.3): perplejidad = 141.2


λ = (0.1, 0.5, 0.4): perplejidad = 162.8
λ = (0.1, 0.3, 0.6): perplejidad = 184.9


La interpolación de bigramas y unigramas supera con amplitud a cualquier modelo add-k. En cambio, añadir el trigrama **empeora** la perplejidad: con este corpus tan pequeño, los conteos de trigramas son tan escasos que sus estimaciones MLE resultan demasiado ruidosas. Con corpus de millones de palabras, el trigrama interpolado sí mejora. (Cuando un contexto nunca se vio, su término simplemente no aporta, así que estas "probabilidades" no suman exactamente 1; las implementaciones rigurosas redistribuyen esa masa, como hace el *backoff* de Katz.)

<a id="generacion-de-texto"></a>
<h2 style="color: #007ACC;">Generación de texto</h2>

Un modelo de lenguaje también **genera** texto: muestreamos la primera palabra de $P(w \mid \texttt{<s>})$, luego la siguiente condicionada a la anterior, y así hasta generar `</s>`. Usamos los conteos MLE (sin suavizado) para muestrear solo continuaciones vistas.

La **temperatura** $T$ controla la diversidad: elevamos las probabilidades a $1/T$ y renormalizamos. Con $T < 1$ el modelo elige casi siempre lo más probable (texto repetitivo); con $T > 1$ se arriesga más (texto más variado, pero menos coherente).

In [14]:
def construir_continuaciones(oraciones, n):
    cont = defaultdict(Counter)
    for s in oraciones:
        for g in ngramas(con_marcadores(s, n), n):
            cont[g[:-1]][g[-1]] += 1
    return cont


def generar(cont, n, temperatura=1.0, max_palabras=30, rng=None):
    rng = rng or np.random.default_rng()
    contexto = ("<s>",) * (n - 1)
    palabras = []
    for _ in range(max_palabras):
        opciones = cont[contexto]
        candidatas = list(opciones)
        p = np.array([opciones[w] for w in candidatas], dtype=float) ** (1 / temperatura)
        palabra = candidatas[rng.choice(len(candidatas), p=p / p.sum())]
        if palabra == "</s>":
            break
        palabras.append(palabra)
        contexto = (contexto + (palabra,))[1:] if n > 1 else ()
    return " ".join(palabras)


rng = np.random.default_rng(SEED)
for n in [1, 2, 3]:
    cont = construir_continuaciones(train, n)
    print(f"--- {n}-gramas ---")
    for _ in range(2):
        print("  " + generar(cont, n, rng=rng))

--- 1-gramas ---


  peores a lateral derecha . inorgánico dictadura generado
  empresa de singulares hace quizá las en ) y centroamericanos algunos coincidencias de the_memory_of_certains_persons obstáculos al_fin , un y , estos países comisión_de_interior_del_congreso el de un ,
--- 2-gramas ---
  - .
  pero nadie por el ejercicio anterior , porque ya no se disputa de minas de queda para alojar equipos catalanes en los periódicos , dijo el sector que en los


--- 3-gramas ---
  en su autobiografía consigna también zznm un recuerdo " para que el anterior , situándose en 75.412 millones de dólares en premios .
  la susodicha era más alta tiene que acabar reconociéndola " .


In [15]:
cont3 = construir_continuaciones(train, 3)
for T in [0.5, 1.0, 2.0]:
    print(f"--- temperatura {T} ---")
    for _ in range(2):
        print("  " + generar(cont3, 3, temperatura=T, rng=rng))

--- temperatura 0.5 ---
  el presidente de la ue y el título de estas cuartillas otro título posterior de juan , sub_rosa , en el que se caracteriza por una travesía de sant_boi_de_llobregat (
  - - en el que se crease algo parecido nos sucede a todos los puertos europeos .
--- temperatura 1.0 ---
  el protagonista de la civilización , de 23 países celebran a_partir_de hoy lunes productos petroleros a montenegro y a craioveanu , del estado aseguró que su configuración será muy distinta
  reciclaje , ahorro , aprovechamiento de " confuso , el ideal de debilitar al_máximo nuestras inclinaciones sería lo peor que puede ser mejor para afrontar la liga profesional : la
--- temperatura 2.0 ---
  a_la_larga , ella lo había hecho con el logotipo de marbella fueron con el ánimo del partido_popular " prepotencia , con terribles efectos para la ética de la política de
  urzaiz marcó en el marcador , pero serán sancionados económicamente si no la perdió y que si no hacemos otra - , mi amigo - y - 

- Los **unigramas** producen palabras sueltas sin relación.
- Los **bigramas** ya forman fragmentos gramaticales locales (artículo + sustantivo, preposición + artículo), pero la oración pierde el hilo rápidamente.
- Los **trigramas** son más fluidos, pero con tan pocos datos tienden a **copiar** fragmentos enteros del corpus de entrenamiento: muchos contextos de dos palabras solo tienen una continuación posible.
- Con temperatura baja el texto es más "seguro" y repetitivo; con temperatura alta salta entre temas sin coherencia.

Esta es la limitación fundamental de los N-gramas: una ventana de contexto corta y fija, y conteos que no generalizan entre palabras similares (*"el presidente dijo"* no le enseña nada sobre *"la ministra dijo"*). En el módulo 3 veremos modelos neuronales que resuelven ambos problemas con *embeddings* y redes recurrentes. Antes, en el siguiente cuaderno, usaremos la API de clasificación de NLTK con características diseñadas a mano.

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Bird, S., Klein, E. & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. [nltk.org/book](https://www.nltk.org/book/)
- Chen, S. F. & Goodman, J. (1999). An empirical study of smoothing techniques for language modeling. *Computer Speech & Language*, 13(4).
- Corpus CESS-ESP. Martí, M. A. et al. (2007). *CESS-ECE: A Multilingual and Multilevel Annotated Corpus*.